# NBA Game Predictor

Pre-game winner picks and a live win-probability model for NBA games, built from 35,527 regular-season games (1996-97 to 2025-26) of play-by-play data.

**What this notebook does**

1. Downloads the raw data, pinned to fixed commits and checked against SHA-256 checksums: play-by-play and shot charts from [shufinskiy/nba_data](https://github.com/shufinskiy/nba_data), and closing betting lines from the Sportsbook Review archive in [kyleskom/NBA-Machine-Learning-Sports-Betting](https://github.com/kyleskom/NBA-Machine-Learning-Sports-Betting).
2. Builds game results and pre-game features: Elo ratings, rest and back-to-backs, season and last-10 form, league-wide home-court edge.
3. Trains the pre-game model and grades it on the four most recent seasons (2022-23 to 2025-26), against Elo, the home team and the betting market.
4. Trains the live win-probability model (score, clock and the pre-game view, every 15 seconds of every game).
5. Registers both models in Fabric (MLflow) and saves the report tables to the `NBA_Lakehouse` Lakehouse.

Every season is predicted by a model trained only on earlier seasons, so the reported accuracy is what the model would have achieved in real time. Runs in about 10 minutes on a starter pool.

*Emmanuel Jibiri · Data Analyst · Houston, TX*

In [ ]:
%%configure -f
{
    "defaultLakehouse": {"name": "NBA_Lakehouse"}
}

## Settings
Paths, seasons and the pinned data sources (the same `config.py` the repository uses).

In [ ]:
import json
import types
from pathlib import Path

ROOT = Path("/tmp/nba")
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "outputs" / "tables"
FIGURES = ROOT / "outputs" / "figures"
FABRIC = ROOT / "fabric"

# Play-by-play source: https://github.com/shufinskiy/nba_data (stats.nba.com play-by-play v3
# and shot charts, one archive per season). Betting lines: OddsData.sqlite from
# https://github.com/kyleskom/NBA-Machine-Learning-Sports-Betting (Sportsbook Review archive).
# Both are pinned to a commit, and every file is checked against src/checksums.json.
PBP_URL = "https://raw.githubusercontent.com/shufinskiy/nba_data/e829d4678be1e075f99e5d41a1c5f97089be446b/datasets/{name}"
ODDS_URL = ("https://raw.githubusercontent.com/kyleskom/NBA-Machine-Learning-Sports-Betting/"
            "8e36b0b72ecc5082aa8976dff742cdf910b3aa36/Data/OddsData.sqlite")
CHECKSUMS = ROOT / "checksums.json"

FIRST_SEASON = 1996          # 1996-97: Elo ratings need a few seasons to settle
LAST_SEASON = 2025           # 2025-26
FIRST_MODEL_SEASON = 2007    # 2007-08: first season with betting lines to compare against

# Time-based evaluation: fit on older seasons, choose settings on validation seasons, report
# on the most recent seasons, which the model never sees while it is being built.
TRAIN_SEASONS = range(2007, 2020)     # 2007-08 .. 2019-20
VALID_SEASONS = range(2020, 2022)     # 2020-21, 2021-22
TEST_SEASONS = range(2022, 2026)      # 2022-23 .. 2025-26

RANDOM_STATE = 42


def season_label(season: int) -> str:
    return f"{season}-{(season + 1) % 100:02d}"

# Expected SHA-256 of every raw file (the download step refuses files that differ)
EXPECTED_FILES = {
 "OddsData.sqlite": "4a803d41cd4597c1d898c0b131a58763dee8b64140be8f40c29b73c852e162d9",
 "nbastatsv3_1996.tar.xz": "393f4c34fab62fd20f5550301ddbdca2ff041fa1c11bc2a7e2a734af7537dc72",
 "nbastatsv3_1997.tar.xz": "690815bed5f67f8e26d744aa16809bd5a4d16b3a216dc737608d727557f8416c",
 "nbastatsv3_1998.tar.xz": "75c9beb763aef4da55d2d87fd5c947f54c43ef10d6df02ce33fbc308160a6f4a",
 "nbastatsv3_1999.tar.xz": "84e9fcd6ba45d9ff624bc7336d06362d675808243ecaab3e4221c11d73bc0fd4",
 "nbastatsv3_2000.tar.xz": "02917169feb060bf1cfcdf4ded762ae1489af8e2b963070b2f1096206cad5517",
 "nbastatsv3_2001.tar.xz": "adcaeee9423c4404404bccf92c85fc8c774db88b957c6ec12df2051fb8756187",
 "nbastatsv3_2002.tar.xz": "9d54187e3e120a7be83c1345d73d0e46f5bb84a3a1f4437c71ab2a26b2fe211e",
 "nbastatsv3_2003.tar.xz": "afa45748c3db1511c1171c68385cda0b70b0d6ff5a2e38d44b953fdde334b84e",
 "nbastatsv3_2004.tar.xz": "9796c7b5175b212bfc6f5bf16ceb22eba1a16e8c4c5c72243d720cba37aa19a3",
 "nbastatsv3_2005.tar.xz": "c56c3e2f08e1c3a66a4774069e55f7c7bf0e738d9193ab537c7d6bbae4f2761c",
 "nbastatsv3_2006.tar.xz": "b6653696e22dc55a7af122a753439d31aee0365ef79bb9ec3c312f521281aee0",
 "nbastatsv3_2007.tar.xz": "2da43e84b9500bc6d0aea617e89a696f223627249bdbb6f30a9648c1a2c8f963",
 "nbastatsv3_2008.tar.xz": "dcf8dc95190f722d1a724bfdae85a4222e6b705e7e145a5fa7e48b7785d587f6",
 "nbastatsv3_2009.tar.xz": "afd54eb8508af5d38184b2d123121affc9799d19daf9be2108b43671626e97b0",
 "nbastatsv3_2010.tar.xz": "8cea2fbdaa57261b7adaf1f439c9b050b70d68c6f645c21209b21f0bcd8d71fb",
 "nbastatsv3_2011.tar.xz": "17d12c3c560d8a44a656e55b1e904d8028c5658f0c699df6f6b4441af40f1a41",
 "nbastatsv3_2012.tar.xz": "35544361ed9c850d55ea6d092d7228a44398d9547c35b69f2de7b4581f9e1436",
 "nbastatsv3_2013.tar.xz": "c9c070f31126eed1336ba2cffd08af9888fa2829ccdd3069b588ea49f79eac59",
 "nbastatsv3_2014.tar.xz": "76a5398235900fd5b914eaeb1464e76fc05517184d04c503404a16a44d14f741",
 "nbastatsv3_2015.tar.xz": "7bbb1e8a1512dd2877be97cb73d990c424a2f7488137ca5d7a6285e2afea873e",
 "nbastatsv3_2016.tar.xz": "6c94890f389abe5af085ff19a26a80e7fe0d205c55dc201aefaa078a887540f4",
 "nbastatsv3_2017.tar.xz": "e9fedc33337e6a34dc4373bc5138e742e6bbcf09477b34568d79588dead7e62c",
 "nbastatsv3_2018.tar.xz": "a8c05607863b0bb0da51ed65fc89866cbb5c1954f7951ce00ed05f2ad178097d",
 "nbastatsv3_2019.tar.xz": "3fc651e4c294e231723fa097dca2cf938dd868ee2b9a6111f22ab9c0728e89dc",
 "nbastatsv3_2020.tar.xz": "8bbd2406d741f8247db9eec222bb1655ea699edb60907d3851a048b8ea7a798d",
 "nbastatsv3_2021.tar.xz": "f44ec8a4a9864732fca66943ae4d06a06ef98ba3932571c15a23ecae644f84c4",
 "nbastatsv3_2022.tar.xz": "42c24abf0abbc0ccef00addaba4b8c76df230a0ff3c0e5b301353307941fd04c",
 "nbastatsv3_2023.tar.xz": "3a6976219b87fb50cecf8b2c54cb2af110f2f013ed8368d694035806802c8b04",
 "nbastatsv3_2024.tar.xz": "38fb8bfb43be8db3d47576600f32f25d89cc3cc05ebac77a51f4db42faf0b841",
 "nbastatsv3_2025.tar.xz": "790a01ea3f0b2f8335d96bd687e21501551fc8cdea3ada83457ee3208a940671",
 "shotdetail_1996.tar.xz": "3828d0ba32f768087c46eb20bd30914c1570e0f10aca6dc0d0c92247b7665248",
 "shotdetail_1997.tar.xz": "cd3f2576b8a87f237cbcf28eb03afb7fda4eef646e471f5596715b35795b2648",
 "shotdetail_1998.tar.xz": "c71e14d1d6e8f29cd91bfaebace4c378174ca4ab30700ee87fb51ef8656b53d2",
 "shotdetail_1999.tar.xz": "e92c11cf10e20d7b458753dd2a9c650ff6c8a1fb4c08bd41775cda5a0773b595",
 "shotdetail_2000.tar.xz": "09c7eb4194f8134ed0ac1e72e6773114b014f9c9eca71079bcadb69bb5f16c8f",
 "shotdetail_2001.tar.xz": "2a8a136e546806a4e6f4f0a23a3353b874d3a2b5ea32bcef73bed20a4655a272",
 "shotdetail_2002.tar.xz": "81cb202ca453da71d0922fe981b44dd2fdbfb227cb49628fdb43e1a374bb457a",
 "shotdetail_2003.tar.xz": "558be852d48c95ec6d3ff1b6b9e172b3447cc3e56fa946caa171bfed6832f9b0",
 "shotdetail_2004.tar.xz": "4159f7e2308f7924757241d125ddfd8784ac8b17626acf08de5d1bb7960aa776",
 "shotdetail_2005.tar.xz": "b6b10669e731b8f97f9596d5f07afbf1b11c76e3997d5f65d260c2f3e7bacc3f",
 "shotdetail_2006.tar.xz": "7c238aee908a59bb707b34da321fafcb63d677fc10349997ba7c1fb7ded96ecf",
 "shotdetail_2007.tar.xz": "067489fce1075f51cbce32f7b59a2f5e2c406a95d8f46f830a0a492677d0ceea",
 "shotdetail_2008.tar.xz": "4cb881e5393893bce7c672c6f2d2839a6f19de478b6ce27fcf8cf75718ffbe3a",
 "shotdetail_2009.tar.xz": "4f8613a0a4842acc6363a9f64360738e212ec2ca1f8f9423412e8f4e69cd134a",
 "shotdetail_2010.tar.xz": "e8a42d4fede746bb30abcb9df99e489af12f90499475554fdd821c3bd9fe6f2c",
 "shotdetail_2011.tar.xz": "15ba3f10dd8ac286a557fc8df6116a4a2fd6ca3b66c1d1b353b650e2dbe69f1b",
 "shotdetail_2012.tar.xz": "c626b0c04e6dc425fce11514a7a86e60dc1261565a1fa8b289f9bb31ff62c5a4",
 "shotdetail_2013.tar.xz": "de5098ccc7cd3f95c128f77fb29e7c3fcfb1987b368e93c6e121a03fbecb2de0",
 "shotdetail_2014.tar.xz": "e2bf753050854e1eb99f6d5c5d690cdcfed4adae216726d330ef6adb3cd21513",
 "shotdetail_2015.tar.xz": "ac48fad67cabf79f8a5a726aa55ea4a8c793a85eaa3835d38b46f2d0e37495a3",
 "shotdetail_2016.tar.xz": "0681285a0bf08a1ef84b24895d2a4bb709f2d192ba9134e3600567c593dacd96",
 "shotdetail_2017.tar.xz": "a62d25aa9c9190477f350f06a9a2d808c7908a6720786970319a3b28af184e81",
 "shotdetail_2018.tar.xz": "c658890976559b69dff44ed08352baf64b935514506e726d21f34950ad5bfe9d",
 "shotdetail_2019.tar.xz": "0c6002cbc8bc379ecc0a2e0f57defb2e9804be8240aa0234242002ec5dd124fe",
 "shotdetail_2020.tar.xz": "02916ac0a4a0c8dcb1e115ed08ddb49b360305f98f4882c8933a2998b747487d",
 "shotdetail_2021.tar.xz": "5b56b6833d179fdef9c76223064c7923717b1add83570becf494a3f1d0a785f2",
 "shotdetail_2022.tar.xz": "62533f4e444ef0fc3354cdf6bbc04e249eb14ba9f444e6cd247c3e95d235e46d",
 "shotdetail_2023.tar.xz": "303e71f967c199568b345cd73a75e595f1f99532cca3e6ce59afb286b63f840d",
 "shotdetail_2024.tar.xz": "950100f9c84fe750ae2ac8eaeb82d24cdfb9f60a9ed04ceaa66f64789d7c216e",
 "shotdetail_2025.tar.xz": "dcb6c29edda8463132ab62f63118b5cee80d751bcbee9ad98fa38ebe12e6d33c"
}

for _d in (ROOT, RAW, PROCESSED, TABLES, FIGURES, FABRIC):
    _d.mkdir(parents=True, exist_ok=True)
CHECKSUMS.write_text(json.dumps(EXPECTED_FILES))
C = types.SimpleNamespace(**{k: v for k, v in dict(globals()).items() if k.isupper() or k == 'season_label'})
print('Working folder:', ROOT)

import mlflow
mlflow.autolog(disable=True)   # Fabric logs every model fit by default; only the two final models are logged, at the end

## Step 0 - Download the raw data (about 300 MB) and check every file against its checksum.

In [ ]:
import hashlib
import json
import urllib.request
from concurrent.futures import ThreadPoolExecutor



def fetch(name: str, expected: str) -> str:
    path = C.RAW / name
    if path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected:
        return f"ok      {name}"
    url = C.ODDS_URL if name == "OddsData.sqlite" else C.PBP_URL.format(name=name)
    urllib.request.urlretrieve(url, path)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != expected:
        raise RuntimeError(f"{name}: checksum {digest[:12]}... does not match the pinned file")
    return f"fetched {name}"


def run_download() -> None:
    C.RAW.mkdir(parents=True, exist_ok=True)
    sums = json.loads(C.CHECKSUMS.read_text())
    with ThreadPoolExecutor(max_workers=8) as pool:
        for line in pool.map(lambda kv: fetch(*kv), sums.items()):
            print(line)


run_download()

## Step 1 - Turn NBA play-by-play into game results, scoring timelines and box scores.

Input  : data/raw/nbastatsv3_<season>.tar.xz (play-by-play) and shotdetail_<season>.tar.xz
         (game dates, home/away teams) from github.com/shufinskiy/nba_data
Output : data/processed/games.csv.gz      one row per regular-season game
         data/processed/scoring.csv.gz    every scoring play: game, second of play, score
         data/processed/box.csv.gz        full-game team box score (used only for the
                                          leakage demonstration in step 3)

In [ ]:
import sys
import tarfile
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd


PBP_COLS = ["gameId", "actionNumber", "clock", "period", "teamTricode", "location",
            "scoreHome", "scoreAway", "actionType", "subType", "isFieldGoal", "shotResult",
            "description"]


def read_tar_csv(path, usecols=None):
    with tarfile.open(path, "r:xz") as tar:
        member = next(m for m in tar.getmembers() if m.name.endswith(".csv"))
        return pd.read_csv(tar.extractfile(member), usecols=usecols, low_memory=False)


def clock_seconds(clock: pd.Series) -> pd.Series:
    """'PT11M36.00S' -> seconds left in the period."""
    parts = clock.str.extract(r"PT(\d+)M([\d.]+)S").astype(float)
    return parts[0] * 60 + parts[1]


def elapsed_seconds(period: pd.Series, left: pd.Series) -> pd.Series:
    """Seconds of game time played so far (4 x 12-minute quarters, then 5-minute overtimes)."""
    regulation = (period.clip(upper=4) - 1) * 720 + np.where(period <= 4, 720 - left, 720)
    overtime = np.where(period > 4, (period - 5) * 300 + (300 - left), 0)
    return regulation + overtime


def parse_season(season: int):
    pbp = read_tar_csv(C.RAW / f"nbastatsv3_{season}.tar.xz", usecols=PBP_COLS)
    shots = read_tar_csv(C.RAW / f"shotdetail_{season}.tar.xz",
                         usecols=["GAME_ID", "GAME_DATE", "HTM", "VTM"])
    pbp = pbp[pbp["gameId"].astype(str).str.startswith("2")]          # regular season only
    pbp = pbp.sort_values(["gameId", "period", "actionNumber"], kind="stable")
    pbp["sec"] = elapsed_seconds(pbp["period"], clock_seconds(pbp["clock"]))

    # Scoring timeline: rows where the score changed
    sc = pbp.dropna(subset=["scoreHome", "scoreAway"])[["gameId", "sec", "period", "scoreHome", "scoreAway"]]
    sc = sc.astype({"scoreHome": int, "scoreAway": int})
    sc = sc[(sc[["scoreHome", "scoreAway"]].diff().abs().sum(axis=1) > 0)
            | (sc["gameId"] != sc["gameId"].shift())]
    sc = sc[(sc["scoreHome"] + sc["scoreAway"]) > 0]

    # Teams and final score
    side = (pbp.dropna(subset=["teamTricode", "location"])
               .groupby(["gameId", "location"])["teamTricode"].agg(lambda s: s.mode().iloc[0])
               .unstack())
    final = sc.groupby("gameId").agg(home_pts=("scoreHome", "last"), away_pts=("scoreAway", "last"),
                                     periods=("period", "max"))
    dates = shots.drop_duplicates("GAME_ID").set_index("GAME_ID")
    games = final.join(side.rename(columns={"h": "home", "v": "away"})).join(dates, how="left")
    games["season"] = season
    games = games.reset_index().rename(columns={"gameId": "game_id", "GAME_DATE": "date"})

    # Full-game box score per team (only for the leakage demo: it is known after the game)
    p = pbp[pbp["location"].isin(["h", "v"])]
    fg = p["isFieldGoal"] == 1
    made = p["shotResult"] == "Made"
    three = p["description"].fillna("").str.contains("3PT")
    box = pd.DataFrame({
        "game_id": p["gameId"], "side": p["location"],
        "fga": fg, "fgm": fg & made,
        "fg3a": fg & three, "fg3m": fg & made & three,
        "fta": p["actionType"] == "Free Throw",
        "ftm": (p["actionType"] == "Free Throw") & ~p["description"].fillna("").str.contains("MISS"),
        "reb": p["actionType"] == "Rebound",
        "tov": p["actionType"] == "Turnover",
        "fouls": p["actionType"] == "Foul",
    }).groupby(["game_id", "side"]).sum().reset_index()
    box["season"] = season
    sc = sc.rename(columns={"gameId": "game_id", "scoreHome": "home", "scoreAway": "away"})
    return games, sc[["game_id", "sec", "home", "away"]], box


def run_parse_pbp() -> None:
    seasons = range(C.FIRST_SEASON, C.LAST_SEASON + 1)
    with ProcessPoolExecutor(max_workers=2) as pool:
        results = list(pool.map(parse_season, seasons))
    games = pd.concat([r[0] for r in results], ignore_index=True)
    scoring = pd.concat([r[1] for r in results], ignore_index=True)
    box = pd.concat([r[2] for r in results], ignore_index=True)

    # Basic integrity checks
    games["date"] = pd.to_datetime(games["date"].astype("Int64").astype(str), format="%Y%m%d", errors="coerce")
    bad = games[games[["home", "away", "date"]].isna().any(axis=1) | (games["home_pts"] == games["away_pts"])]
    if len(bad):
        print(f"Dropping {len(bad)} games with missing teams/dates or tied final scores")
    games = games.drop(bad.index)
    games["home_win"] = (games["home_pts"] > games["away_pts"]).astype(int)
    games = games.sort_values(["date", "game_id"]).reset_index(drop=True)

    C.PROCESSED.mkdir(parents=True, exist_ok=True)
    games.to_csv(C.PROCESSED / "games.csv.gz", index=False)
    scoring[scoring["game_id"].isin(games["game_id"])].to_csv(C.PROCESSED / "scoring.csv.gz", index=False)
    box[box["game_id"].isin(games["game_id"])].to_csv(C.PROCESSED / "box.csv.gz", index=False)
    per_season = games.groupby("season").agg(games=("game_id", "size"), home_win=("home_win", "mean"))
    print(per_season.round(3).to_string())
    print(f"{len(games):,} games, {len(scoring):,} scoring plays")


run_parse_pbp()

## Step 2 - Pre-game features and betting lines for every game.

Everything here is known before tip-off:
* Elo ratings (FiveThirtyEight's NBA method: K=20, home court worth 100 Elo points,
  margin-of-victory multiplier, 25% reversion to the mean between seasons)
* rest days and back-to-backs
* season-to-date point differential and last-10-games form, shrunk toward zero early on
* the league-wide home win rate over the previous 365 days (home court has shrunk since 2000)
* the closing moneyline and spread, used only as a benchmark, never as a model input

Output: data/processed/features.csv.gz (one row per game)

In [ ]:
import sqlite3

import numpy as np
import pandas as pd


# Franchise continuity for ratings: relocated teams keep their rating.
FRANCHISE = {"SEA": "OKC", "NJN": "BKN", "VAN": "MEM", "NOH": "NOP", "NOK": "NOP", "CHH": "NOP"}
EXPANSION = {("CHA", 2004)}            # Charlotte Bobcats started in 2004-05

ODDS_NAMES = {
    "Atlanta Hawks": "ATL", "Boston Celtics": "BOS", "Brooklyn Nets": "BKN", "New Jersey Nets": "BKN",
    "Charlotte Bobcats": "CHA", "Charlotte Hornets": "CHA", "Chicago Bulls": "CHI",
    "Cleveland Cavaliers": "CLE", "Dallas Mavericks": "DAL", "Denver Nuggets": "DEN",
    "Detroit Pistons": "DET", "Golden State Warriors": "GSW", "Houston Rockets": "HOU",
    "Indiana Pacers": "IND", "LA Clippers": "LAC", "Los Angeles Clippers": "LAC",
    "Los Angeles Lakers": "LAL", "Memphis Grizzlies": "MEM", "Miami Heat": "MIA",
    "Milwaukee Bucks": "MIL", "Minnesota Timberwolves": "MIN", "New Orleans Pelicans": "NOP",
    "New Orleans Hornets": "NOP", "New York Knicks": "NYK", "Oklahoma City Thunder": "OKC",
    "Seattle SuperSonics": "OKC", "Orlando Magic": "ORL", "Philadelphia 76ers": "PHI",
    "Phoenix Suns": "PHX", "Portland Trail Blazers": "POR", "Sacramento Kings": "SAC",
    "San Antonio Spurs": "SAS", "Toronto Raptors": "TOR", "Utah Jazz": "UTA",
    "Washington Wizards": "WAS",
}

ELO_K, ELO_HOME, ELO_MEAN, ELO_CARRY = 20.0, 100.0, 1505.0, 0.75


def elo_ratings(games: pd.DataFrame) -> pd.DataFrame:
    """Pre-game Elo for both teams, updated game by game in date order."""
    rating, last_season = {}, {}
    pre_h, pre_a = np.empty(len(games)), np.empty(len(games))
    for i, g in enumerate(games.itertuples(index=False)):
        for team in (g.home_f, g.away_f):
            if team not in rating:
                rating[team] = 1300.0 if (team, g.season) in EXPANSION else 1500.0
            elif last_season[team] != g.season:          # new season: revert toward the mean
                rating[team] = ELO_CARRY * rating[team] + (1 - ELO_CARRY) * ELO_MEAN
            last_season[team] = g.season
        rh, ra = rating[g.home_f], rating[g.away_f]
        pre_h[i], pre_a[i] = rh, ra
        diff = rh + ELO_HOME - ra
        p_home = 1 / (1 + 10 ** (-diff / 400))
        margin = g.home_pts - g.away_pts
        winner_diff = diff if margin > 0 else -diff
        mult = (abs(margin) + 3) ** 0.8 / (7.5 + 0.006 * winner_diff)
        shift = ELO_K * mult * ((margin > 0) - p_home)
        rating[g.home_f] = rh + shift
        rating[g.away_f] = ra - shift
    return pd.DataFrame({"elo_home": pre_h, "elo_away": pre_a}, index=games.index)


def team_form(games: pd.DataFrame) -> pd.DataFrame:
    """Rest, season-to-date and last-10 point differential for each side, before the game."""
    long = pd.concat([
        pd.DataFrame({"game_id": games["game_id"], "date": games["date"], "season": games["season"],
                      "team": games["home_f"], "side": "home",
                      "diff": games["home_pts"] - games["away_pts"]}),
        pd.DataFrame({"game_id": games["game_id"], "date": games["date"], "season": games["season"],
                      "team": games["away_f"], "side": "away",
                      "diff": games["away_pts"] - games["home_pts"]}),
    ]).sort_values(["team", "date", "game_id"], kind="stable")
    grp = long.groupby(["team", "season"], sort=False)
    long["games_played"] = grp.cumcount()
    long["std_diff_sum"] = grp["diff"].cumsum() - long["diff"]
    # Shrink early-season averages toward 0 (as if the team had 5 average games already)
    long["season_diff"] = long["std_diff_sum"] / (long["games_played"] + 5)
    long["last10_diff"] = grp["diff"].transform(lambda s: s.shift().rolling(10, min_periods=1).mean()).fillna(0)
    prev_date = long.groupby("team", sort=False)["date"].shift()
    long["rest_days"] = (long["date"] - prev_date).dt.days.clip(upper=5).fillna(5)
    long["b2b"] = (long["rest_days"] == 1).astype(int)
    cols = ["games_played", "season_diff", "last10_diff", "rest_days", "b2b"]
    wide = long.pivot(index="game_id", columns="side", values=cols)
    wide.columns = [f"{c}_{s}" for c, s in wide.columns]
    return wide


def league_home_rate(games: pd.DataFrame) -> pd.Series:
    """Share of home wins in the 365 days before each game day (excludes that day)."""
    daily = games.groupby("date")["home_win"].agg(["sum", "count"])
    roll = daily.rolling("365D", closed="left").sum()
    rate = (roll["sum"] / roll["count"]).reindex(games["date"]).values
    return pd.Series(rate, index=games.index).fillna(0.6)


def ml_to_prob(ml: pd.Series) -> pd.Series:
    """American moneyline -> implied probability ('NL' = no line -> NaN)."""
    ml = pd.to_numeric(ml, errors="coerce")
    return pd.Series(np.where(ml < 0, -ml / (-ml + 100), 100 / (ml + 100)), index=ml.index)


def load_odds() -> pd.DataFrame:
    con = sqlite3.connect(C.RAW / "OddsData.sqlite")
    tables = pd.read_sql("select name from sqlite_master where type='table'", con)["name"]
    frames = []
    for season in range(C.FIRST_MODEL_SEASON, C.LAST_SEASON + 1):
        label = C.season_label(season)
        for name in (f"odds_{label}_new", label, f"odds_{label}"):
            if name in set(tables):
                d = pd.read_sql(f'select * from "{name}"', con)
                d["season"] = season
                frames.append(d)
                break
    odds = pd.concat(frames, ignore_index=True)
    odds["date"] = pd.to_datetime(odds["Date"])
    odds["home_f"] = odds["Home"].map(ODDS_NAMES)
    odds["away_f"] = odds["Away"].map(ODDS_NAMES)
    assert odds[["home_f", "away_f"]].notna().all().all(), set(odds["Home"]) - set(ODDS_NAMES)
    ph, pa = ml_to_prob(odds["ML_Home"]), ml_to_prob(odds["ML_Away"])
    odds["vegas_prob_home"] = ph / (ph + pa)                   # remove the bookmaker's margin
    odds["vegas_spread_home"] = pd.to_numeric(odds["Spread"], errors="coerce")   # expected home margin
    malformed = (pd.to_numeric(odds["ML_Home"], errors="coerce").abs() < 100) | \
                (pd.to_numeric(odds["ML_Away"], errors="coerce").abs() < 100)
    odds.loc[malformed, "vegas_prob_home"] = np.nan
    odds["odds_margin"] = odds["Win_Margin"]
    odds["odds_points"] = odds["Points"]
    return odds[["date", "home_f", "away_f", "vegas_prob_home", "vegas_spread_home",
                 "odds_margin", "odds_points"]]


def run_features() -> None:
    games = pd.read_csv(C.PROCESSED / "games.csv.gz", parse_dates=["date"])
    games["home_f"] = games["home"].replace(FRANCHISE)
    games["away_f"] = games["away"].replace(FRANCHISE)
    games = games.sort_values(["date", "game_id"], kind="stable").reset_index(drop=True)

    games = games.join(elo_ratings(games))
    games["elo_diff"] = games["elo_home"] - games["elo_away"]
    games["elo_prob_home"] = 1 / (1 + 10 ** (-(games["elo_diff"] + ELO_HOME) / 400))
    games = games.merge(team_form(games), left_on="game_id", right_index=True, how="left")
    games["league_home_rate"] = league_home_rate(games)

    # Betting lines: match on date and teams (allow the listing to be one day off)
    odds = load_odds()
    merged = games.merge(odds, on=["date", "home_f", "away_f"], how="left")
    miss = merged["vegas_prob_home"].isna() & (merged["season"] >= C.FIRST_MODEL_SEASON)
    for shift in (-1, 1):
        alt = odds.assign(date=odds["date"] + pd.Timedelta(days=shift))
        fill = games[miss.values].merge(alt, on=["date", "home_f", "away_f"], how="left")
        for col in ["vegas_prob_home", "vegas_spread_home", "odds_margin", "odds_points"]:
            merged.loc[miss.values, col] = merged.loc[miss.values, col].fillna(pd.Series(fill[col].values, index=merged.index[miss.values]))
        miss = merged["vegas_prob_home"].isna() & (merged["season"] >= C.FIRST_MODEL_SEASON)
    merged = merged.drop_duplicates("game_id")

    # Cross-check our final scores against the odds archive's results
    chk = merged.dropna(subset=["odds_margin"])
    agree = ((chk["home_pts"] - chk["away_pts"]) == chk["odds_margin"]).mean()
    model_games = merged[merged["season"] >= C.FIRST_MODEL_SEASON]
    print(f"Betting lines matched for {model_games['vegas_prob_home'].notna().mean():.1%} of games since 2007-08; "
          f"final margins agree with the odds archive in {agree:.2%} of matched games")

    C.PROCESSED.mkdir(parents=True, exist_ok=True)
    merged.drop(columns=["odds_margin", "odds_points"]).to_csv(C.PROCESSED / "features.csv.gz", index=False)
    print(merged.groupby("season")[["vegas_prob_home"]].count().T.to_string())


run_features()

## Step 3 - Pre-game winner model, graded against simple baselines and the betting market.

* Model choice: candidates are fit on 2007-08 .. 2019-20 and compared on 2020-21 and 2021-22.
* Walk-forward predictions: every season from 2010-11 on is predicted by a model trained only
  on earlier seasons, the way it would run for real. The test seasons 2022-23 .. 2025-26
  play no part in choosing or fitting the model that predicts them.
* The 99% trap: the same kind of model fed the final box score (shooting, rebounds,
  turnovers) looks brilliant, but those numbers only exist after the final buzzer.

Outputs (outputs/tables/): pregame_validation.csv, pregame_models.csv, pregame_by_season.csv,
pregame_confidence.csv, pregame_calibration.csv, leakage_demo.csv
and data/processed/pregame_predictions.csv.gz

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


FEATURES = ["elo_diff", "rest_days_home", "rest_days_away", "b2b_home", "b2b_away",
            "season_diff_home", "season_diff_away", "last10_diff_home", "last10_diff_away",
            "games_played_home", "games_played_away", "league_home_rate"]


def candidates():
    return {
        "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000)),
        "Gradient boosting": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=300,
                                                            l2_regularization=1.0,
                                                            random_state=C.RANDOM_STATE),
    }


def choose(val: pd.DataFrame, simple: str, margin: float = 0.002) -> str:
    """Keep the simpler model unless another one lowers validation log loss by more than
    `margin`. Near-ties then can't flip the choice between machines or library versions."""
    best = val.loc[val["log_loss"].idxmin()]
    simple_loss = val.loc[val["model"] == simple, "log_loss"].iloc[0]
    return best["model"] if best["log_loss"] < simple_loss - margin else simple


def scores(y, p) -> dict:
    y, p = np.asarray(y), np.asarray(p)
    return {"games": len(y), "accuracy": accuracy_score(y, p > 0.5), "log_loss": log_loss(y, p, labels=[0, 1]),
            "brier": brier_score_loss(y, p), "auc": roc_auc_score(y, p)}


def run_pregame_model() -> None:
    df = pd.read_csv(C.PROCESSED / "features.csv.gz", parse_dates=["date"])
    df = df[df["season"] >= C.FIRST_MODEL_SEASON].reset_index(drop=True)
    y = df["home_win"].values
    train = df["season"].isin(C.TRAIN_SEASONS).values
    valid = df["season"].isin(C.VALID_SEASONS).values

    # ---- Model choice on the validation seasons ---------------------------------------------
    rows = []
    for name, m in candidates().items():
        m.fit(df.loc[train, FEATURES], y[train])
        rows.append({"model": name, **scores(y[valid], m.predict_proba(df.loc[valid, FEATURES])[:, 1])})
    val = pd.DataFrame(rows)
    val.round(4).to_csv(C.TABLES / "pregame_validation.csv", index=False)
    choice = choose(val, simple="Logistic regression")
    print(val.round(4).to_string(index=False), f"\nChosen: {choice}")

    # ---- Walk-forward predictions -------------------------------------------------------------
    preds = []
    for season in range(2010, C.LAST_SEASON + 1):
        fit_on = (df["season"] < season).values
        m = candidates()[choice].fit(df.loc[fit_on, FEATURES], y[fit_on])
        part = df[df["season"] == season]
        preds.append(part[["game_id"]].assign(model_prob_home=m.predict_proba(part[FEATURES])[:, 1]))
    wf = df.merge(pd.concat(preds), on="game_id")
    wf[["game_id", "season", "date", "home", "away", "home_pts", "away_pts", "home_win",
        "elo_home", "elo_away", "elo_prob_home", "vegas_prob_home", "vegas_spread_home",
        "model_prob_home"]].to_csv(C.PROCESSED / "pregame_predictions.csv.gz", index=False)

    # ---- Test seasons: model vs. baselines vs. the betting market --------------------------
    t = wf[wf["season"].isin(C.TEST_SEASONS)]
    lined = t.dropna(subset=["vegas_prob_home"])
    table = pd.DataFrame([
        {"model": "Our pre-game model", "games": "all", **scores(t["home_win"], t["model_prob_home"])},
        {"model": "Elo rating alone", "games": "all", **scores(t["home_win"], t["elo_prob_home"])},
        # constant 0.5 + tiny edge = "home team wins"; its log loss is that of a coin flip
        {"model": "Always pick the home team", "games": "all", **scores(t["home_win"], np.full(len(t), 0.501))},
        {"model": "Our pre-game model", "games": "with betting lines", **scores(lined["home_win"], lined["model_prob_home"])},
        {"model": "Betting market favourite (closing moneyline)", "games": "with betting lines",
         **scores(lined["home_win"], lined["vegas_prob_home"])},
    ])
    table.round(4).to_csv(C.TABLES / "pregame_models.csv", index=False)
    print(table.round(3).to_string(index=False))
    agree = (lined["model_prob_home"] > 0.5) == (lined["vegas_prob_home"] > 0.5)
    print(f"Same pick as the betting market in {agree.mean():.1%} of test games with lines")

    # ---- Confidence and calibration on the test seasons -----------------------------------
    conf = np.maximum(t["model_prob_home"], 1 - t["model_prob_home"])
    right = (t["model_prob_home"] > 0.5) == (t["home_win"] == 1)
    cov = pd.DataFrame([{"min_confidence": c, "share_of_games": (conf >= c).mean(),
                         "accuracy": right[conf >= c].mean(), "games": int((conf >= c).sum())}
                        for c in np.round(np.arange(0.50, 0.91, 0.05), 2)])
    cov.round(4).to_csv(C.TABLES / "pregame_confidence.csv", index=False)
    print(cov.round(3).to_string(index=False))
    bins = pd.cut(t["model_prob_home"], np.linspace(0, 1, 11))
    cal = (t.groupby(bins, observed=True)
             .agg(games=("home_win", "size"), predicted=("model_prob_home", "mean"), actual=("home_win", "mean"))
             .reset_index(drop=True))
    cal.round(4).to_csv(C.TABLES / "pregame_calibration.csv", index=False)

    # ---- Season by season ---------------------------------------------------------------------
    rows = []
    for season, g in wf.groupby("season"):
        l = g.dropna(subset=["vegas_prob_home"])
        rows.append({
            "season": season, "season_label": C.season_label(season), "games": len(g),
            "model_accuracy": accuracy_score(g["home_win"], g["model_prob_home"] > 0.5),
            "elo_accuracy": accuracy_score(g["home_win"], g["elo_prob_home"] > 0.5),
            "home_team_accuracy": g["home_win"].mean(),
            "games_with_lines": len(l),
            "vegas_accuracy": accuracy_score(l["home_win"], l["vegas_prob_home"] > 0.5),
            "model_accuracy_on_lined_games": accuracy_score(l["home_win"], l["model_prob_home"] > 0.5),
        })
    by_season = pd.DataFrame(rows)
    by_season["split"] = np.where(by_season["season"].isin(C.TEST_SEASONS), "test", "earlier")
    by_season.round(4).to_csv(C.TABLES / "pregame_by_season.csv", index=False)
    print(by_season.round(3).to_string(index=False))

    # ---- The 99% trap: a "model" that reads the final box score ------------------------------
    box = pd.read_csv(C.PROCESSED / "box.csv.gz")
    wide = box.pivot(index="game_id", columns="side")
    wide.columns = [f"{c}_{s}" for c, s in wide.columns]
    stats = pd.DataFrame(index=wide.index)
    for s in ("h", "v"):
        stats[f"fg_pct_{s}"] = wide[f"fgm_{s}"] / wide[f"fga_{s}"]
        for c in ("fg3m", "ftm", "reb", "tov"):
            stats[f"{c}_{s}"] = wide[f"{c}_{s}"]
    leak = df[["game_id", "season", "home_win", "home_pts", "away_pts"]].merge(stats, left_on="game_id", right_index=True)
    fit_on = leak["season"] < min(C.TEST_SEASONS)
    test_on = leak["season"].isin(C.TEST_SEASONS)
    rows = [{"model": "Pre-game model (honest)", "test_accuracy": accuracy_score(t["home_win"], t["model_prob_home"] > 0.5),
             "known_before_tipoff": True}]
    for label, cols in [("Final box score: shooting, rebounds, turnovers", list(stats.columns)),
                        ("Final score", ["home_pts", "away_pts"])]:
        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(leak.loc[fit_on, cols], leak.loc[fit_on, "home_win"])
        rows.append({"model": label, "test_accuracy": accuracy_score(leak.loc[test_on, "home_win"], m.predict(leak.loc[test_on, cols])),
                     "known_before_tipoff": False})
    leak_tbl = pd.DataFrame(rows)
    leak_tbl.round(4).to_csv(C.TABLES / "leakage_demo.csv", index=False)
    print(leak_tbl.round(3).to_string(index=False))

    # The model to use next season: same recipe, fit on every season so far
    deployed = candidates()[choice].fit(df[FEATURES], y)
    ours, vegas = table.iloc[0], table.iloc[4]
    return {"model": deployed, "name": choice, "input_example": df[FEATURES].tail(1000),
            "metrics": {"test_accuracy": ours["accuracy"], "test_log_loss": ours["log_loss"],
                        "test_brier": ours["brier"], "test_auc": ours["auc"],
                        "test_accuracy_same_games_as_market": table.iloc[3]["accuracy"],
                        "market_accuracy": vegas["accuracy"]}}


pregame = run_pregame_model()

## Step 4 - Live win probability: who wins, given the score, the clock and the pre-game odds?

Every game is sampled every 15 seconds of game time. Each moment's features are the home
team's lead, the time left and the pre-game model's probability (whose weight fades as the
game goes on). Fit on 2010-11 .. 2021-22, reported on 2022-23 .. 2025-26.

Outputs (outputs/tables/): live_validation.csv, live_by_minute.csv, live_checkpoints.csv
and data/processed/live_wp.csv.gz (win probability every 15 seconds for the test seasons)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


STEP = 15                    # seconds between snapshots
REGULATION = 48 * 60
FIT_SEASONS = range(2010, 2020)        # fit candidates ...
CHOOSE_SEASONS = range(2020, 2022)     # ... and choose on these, then refit on both
CHECKPOINTS = [("Tip-off", 0), ("End of 1st quarter", 720), ("Halftime", 1440),
               ("End of 3rd quarter", 2160), ("5 minutes left", 2580), ("2 minutes left", 2760),
               ("1 minute left", 2820), ("30 seconds left", 2850), ("15 seconds left", 2865)]


def logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p))


def snapshots(games: pd.DataFrame, scoring: pd.DataFrame) -> pd.DataFrame:
    """One row per game every STEP seconds: lead, time left, pre-game probability."""
    end = REGULATION + 300 * (games["periods"].clip(lower=4) - 4)
    n = (end // STEP).astype(int)
    grid = pd.DataFrame({"game_id": np.repeat(games["game_id"].values, n),
                         "t": np.concatenate([np.arange(k) * STEP for k in n]).astype(float)})
    grid = grid.merge(games[["game_id", "season", "home_win", "model_prob_home"]], on="game_id")
    sc = scoring.sort_values("sec")
    grid = pd.merge_asof(grid.sort_values("t"), sc.rename(columns={"sec": "t"}), on="t", by="game_id",
                         direction="backward")
    grid[["home", "away"]] = grid[["home", "away"]].fillna(0)
    grid["margin"] = grid["home"] - grid["away"]
    ot = grid["t"] >= REGULATION
    grid["remaining"] = np.where(ot, 300 - (grid["t"] - REGULATION) % 300, REGULATION - grid["t"])
    grid["is_ot"] = ot.astype(int)
    grid["prior_logit"] = logit(grid["model_prob_home"])
    return grid.sort_values(["game_id", "t"]).reset_index(drop=True)


def design(d: pd.DataFrame) -> pd.DataFrame:
    """Features for the logistic model: the lead measured against the time left, and the
    pre-game view, which counts for less as the game goes on."""
    rem_min = d["remaining"] / 60
    frac = np.where(d["is_ot"] == 1, 0.0, d["remaining"] / REGULATION)
    return pd.DataFrame({
        "lead_vs_time": d["margin"] / np.sqrt(rem_min + 0.5),
        "lead": d["margin"],
        "prior_x_time": d["prior_logit"] * np.sqrt(frac),
        "prior": d["prior_logit"],
        "is_ot": d["is_ot"],
    })


def live_candidates():
    return {
        "Logistic regression (engineered)": ("design", make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))),
        "Gradient boosting (monotone)": ("raw", HistGradientBoostingClassifier(
            max_depth=6, learning_rate=0.1, max_iter=300, monotonic_cst=[1, 0, 1, 0],
            random_state=C.RANDOM_STATE)),
    }


def live_features(kind, d):
    return design(d) if kind == "design" else d[["margin", "remaining", "prior_logit", "is_ot"]]


def run_live_model() -> None:
    games = pd.read_csv(C.PROCESSED / "pregame_predictions.csv.gz")
    games = games.merge(pd.read_csv(C.PROCESSED / "games.csv.gz", usecols=["game_id", "periods"]), on="game_id")
    scoring = pd.read_csv(C.PROCESSED / "scoring.csv.gz")
    scoring = scoring[scoring["game_id"].isin(games["game_id"])]
    snap = snapshots(games, scoring)
    print(f"{len(snap):,} snapshots from {snap['game_id'].nunique():,} games")
    y = snap["home_win"].values

    fit = snap["season"].isin(FIT_SEASONS).values
    choose = snap["season"].isin(CHOOSE_SEASONS).values
    test = snap["season"].isin(C.TEST_SEASONS).values

    rows = []
    for name, (kind, m) in live_candidates().items():
        m.fit(live_features(kind, snap[fit]), y[fit])
        p = m.predict_proba(live_features(kind, snap[choose]))[:, 1]
        rows.append({"model": name, "log_loss": log_loss(y[choose], p), "brier": brier_score_loss(y[choose], p),
                     "accuracy": accuracy_score(y[choose], p > 0.5)})
    val = pd.DataFrame(rows)
    val.round(4).to_csv(C.TABLES / "live_validation.csv", index=False)
    best = val.loc[val["log_loss"].idxmin()]
    simple = "Logistic regression (engineered)"
    simple_loss = val.loc[val["model"] == simple, "log_loss"].iloc[0]
    choice = best["model"] if best["log_loss"] < simple_loss - 0.002 else simple   # keep it simple on near-ties
    print(val.round(4).to_string(index=False), f"\nChosen: {choice}")

    kind, model = live_candidates()[choice]
    model.fit(live_features(kind, snap[fit | choose]), y[fit | choose])
    t = snap[test].copy()
    t["wp"] = model.predict_proba(live_features(kind, t))[:, 1]
    # Baseline without the pre-game view: the score and clock alone
    base = live_candidates()["Logistic regression (engineered)"][1]
    cols = ["lead_vs_time", "lead", "is_ot"]
    base.fit(design(snap[fit | choose])[cols], y[fit | choose])
    t["wp_score_only"] = base.predict_proba(design(t)[cols])[:, 1]

    reg = t[t["is_ot"] == 0].copy()
    reg["minute"] = (reg["t"] // 60).astype(int)          # minutes played so far, 0..47
    by_min = pd.DataFrame([{
        "minute": minute, "minutes_left": 48 - minute, "snapshots": len(g),
        "accuracy": accuracy_score(g["home_win"], g["wp"] > 0.5),
        "accuracy_score_only": accuracy_score(g["home_win"], g["wp_score_only"] > 0.5),
        "brier": brier_score_loss(g["home_win"], g["wp"]),
    } for minute, g in reg.groupby("minute")])
    by_min.round(4).to_csv(C.TABLES / "live_by_minute.csv", index=False)

    cps = []
    for label, sec in CHECKPOINTS:
        g = reg[reg["t"] == sec]
        cps.append({"moment": label, "seconds_played": sec, "games": len(g),
                    "accuracy": accuracy_score(g["home_win"], g["wp"] > 0.5),
                    "accuracy_score_only": accuracy_score(g["home_win"], g["wp_score_only"] > 0.5),
                    "log_loss": log_loss(g["home_win"], g["wp"], labels=[0, 1])})
    cps = pd.DataFrame(cps)
    cps.round(4).to_csv(C.TABLES / "live_checkpoints.csv", index=False)
    print(cps.round(3).to_string(index=False))

    out = t[["game_id", "t", "remaining", "is_ot", "home", "away", "margin", "wp"]]
    out.assign(wp=out["wp"].round(4)).to_csv(C.PROCESSED / "live_wp.csv.gz", index=False)
    print(f"Saved {len(out):,} live snapshots for {out['game_id'].nunique():,} test games")
    sample = t.sample(1000, random_state=C.RANDOM_STATE)
    return {"model": model, "name": choice, "input_example": live_features(kind, sample),
            "metrics": {f"accuracy_{r.moment.lower().replace(' ', '_')}": r.accuracy for r in cps.itertuples()}}


live = run_live_model()

## Step 5 - Report-ready tables for the Fabric Lakehouse and Power BI.

Output: fabric/tables/*.csv
  games.csv              one row per game since 2010-11: teams, score, the three pre-game
                         probabilities (our model, Elo, betting market) and whether each pick was right
  teams.csv              team names, conference and Elo rating going into 2026-27
  live_win_probability.csv  home win probability every 15 seconds for the 4,920 test games
  model_scorecard.csv, season_accuracy.csv, confidence.csv, live_checkpoints.csv,
  live_by_minute.csv, leakage_demo.csv   summary tables behind the report's charts

In [ ]:
import numpy as np
import pandas as pd


TEAMS = {
    "ATL": ("Atlanta Hawks", "East"), "BOS": ("Boston Celtics", "East"), "BKN": ("Brooklyn Nets", "East"),
    "NJN": ("New Jersey Nets", "East"), "CHA": ("Charlotte Hornets", "East"), "CHI": ("Chicago Bulls", "East"),
    "CLE": ("Cleveland Cavaliers", "East"), "DAL": ("Dallas Mavericks", "West"), "DEN": ("Denver Nuggets", "West"),
    "DET": ("Detroit Pistons", "East"), "GSW": ("Golden State Warriors", "West"), "HOU": ("Houston Rockets", "West"),
    "IND": ("Indiana Pacers", "East"), "LAC": ("LA Clippers", "West"), "LAL": ("Los Angeles Lakers", "West"),
    "MEM": ("Memphis Grizzlies", "West"), "MIA": ("Miami Heat", "East"), "MIL": ("Milwaukee Bucks", "East"),
    "MIN": ("Minnesota Timberwolves", "West"), "NOP": ("New Orleans Pelicans", "West"),
    "NOH": ("New Orleans Hornets", "West"), "NYK": ("New York Knicks", "East"),
    "OKC": ("Oklahoma City Thunder", "West"), "ORL": ("Orlando Magic", "East"),
    "PHI": ("Philadelphia 76ers", "East"), "PHX": ("Phoenix Suns", "West"),
    "POR": ("Portland Trail Blazers", "West"), "SAC": ("Sacramento Kings", "West"),
    "SAS": ("San Antonio Spurs", "West"), "TOR": ("Toronto Raptors", "East"), "UTA": ("Utah Jazz", "West"),
    "WAS": ("Washington Wizards", "East"),
}
REPORT_FRANCHISE = {"NJN": "BKN", "NOH": "NOP"}


def clock_label(t: pd.Series, is_ot: pd.Series) -> pd.Series:
    period = np.where(is_ot == 1, 5 + (t - 2880) // 300, t // 720 + 1).astype(int)
    left = np.where(is_ot == 1, 300 - (t - 2880) % 300, 720 - t % 720).astype(int)
    name = [f"Q{p}" if p <= 4 else f"OT{p - 4}" for p in period]
    return pd.Series(name, index=t.index), pd.Series(period, index=t.index), \
        pd.Series([f"{s // 60}:{s % 60:02d}" for s in left], index=t.index)


def run_export() -> None:
    out = C.FABRIC / "tables"
    out.mkdir(parents=True, exist_ok=True)

    g = pd.read_csv(C.PROCESSED / "pregame_predictions.csv.gz", parse_dates=["date"])
    g["season_label"] = g["season"].map(C.season_label)
    g["split"] = np.where(g["season"].isin(C.TEST_SEASONS), "Test seasons", "Earlier seasons")
    g["winner"] = np.where(g["home_win"] == 1, g["home"], g["away"])
    for src in ("model", "elo", "vegas"):
        p = g[f"{src}_prob_home"]
        g[f"{src}_pick"] = np.where(p.isna(), None, np.where(p > 0.5, g["home"], g["away"]))
        g[f"{src}_correct"] = np.where(p.isna(), np.nan, ((p > 0.5).astype(int) == g["home_win"]).astype(float))
    g["model_confidence"] = np.maximum(g["model_prob_home"], 1 - g["model_prob_home"]).round(4)
    g["upset"] = ((g["model_prob_home"] > 0.5).astype(int) != g["home_win"]).astype(int)
    g["final_score"] = g["away"] + " " + g["away_pts"].astype(str) + " @ " + g["home"] + " " + g["home_pts"].astype(str)
    g["game_label"] = g["date"].dt.strftime("%Y-%m-%d") + "  " + g["away"] + " @ " + g["home"]
    cols = ["game_id", "season", "season_label", "split", "date", "game_label", "home", "away", "home_pts", "away_pts",
            "final_score", "home_win", "winner", "model_prob_home", "model_pick", "model_confidence",
            "model_correct", "elo_prob_home", "elo_pick", "elo_correct", "vegas_prob_home",
            "vegas_spread_home", "vegas_pick", "vegas_correct", "upset"]
    g = g[cols].rename(columns={"date": "game_date"})
    for c in ("model_prob_home", "elo_prob_home", "vegas_prob_home"):
        g[c] = g[c].round(4)
    g.to_csv(out / "games.csv", index=False)

    # Teams with the rating they carry into 2026-27 (end-of-season Elo, 25% back to the mean)
    feats = pd.read_csv(C.PROCESSED / "features.csv.gz", parse_dates=["date"])
    last = feats[feats["season"] == C.LAST_SEASON].sort_values("date")
    long = pd.concat([last[["date", "home_f", "elo_home"]].set_axis(["date", "team", "elo"], axis=1),
                      last[["date", "away_f", "elo_away"]].set_axis(["date", "team", "elo"], axis=1)])
    end_elo = long.sort_values("date").groupby("team")["elo"].last()
    teams = pd.DataFrame([{"team": k, "team_name": v[0], "conference": v[1],
                           "franchise": REPORT_FRANCHISE.get(k, k)} for k, v in TEAMS.items()])
    teams["elo_end_2025_26"] = teams["franchise"].map(end_elo).round(1)
    teams["elo_start_2026_27"] = (0.75 * teams["elo_end_2025_26"] + 0.25 * 1505).round(1)
    teams.loc[teams["team"] != teams["franchise"], ["elo_end_2025_26", "elo_start_2026_27"]] = np.nan
    teams.to_csv(out / "teams.csv", index=False)

    live = pd.read_csv(C.PROCESSED / "live_wp.csv.gz")
    period_name, period, clock = clock_label(live["t"], live["is_ot"])
    live = pd.DataFrame({
        "game_id": live["game_id"], "seconds_played": live["t"].astype(int),
        "minutes_played": (live["t"] / 60).round(2), "period": period, "period_name": period_name,
        "clock": clock, "home_score": live["home"].astype(int), "away_score": live["away"].astype(int),
        "home_lead": live["margin"].astype(int), "home_win_prob": live["wp"],
    })
    live.to_csv(out / "live_win_probability.csv", index=False)

    T = lambda n: pd.read_csv(C.TABLES / f"{n}.csv")
    T("pregame_models").to_csv(out / "model_scorecard.csv", index=False)
    T("pregame_by_season").to_csv(out / "season_accuracy.csv", index=False)
    T("pregame_confidence").to_csv(out / "confidence.csv", index=False)
    T("live_checkpoints").to_csv(out / "live_checkpoints.csv", index=False)
    T("live_by_minute").to_csv(out / "live_by_minute.csv", index=False)
    T("leakage_demo").to_csv(out / "leakage_demo.csv", index=False)
    for f in sorted(out.glob("*.csv")):
        print(f"{f.name:28s} {sum(1 for _ in open(f)) - 1:>9,} rows  {f.stat().st_size / 1e6:6.2f} MB")


run_export()

## Register both models in Fabric

In [ ]:
import mlflow
from mlflow.models.signature import infer_signature

mlflow.autolog(disable=True)                 # the walk-forward loop fits many models; log only these
mlflow.set_experiment("nba-game-predictor")


def register(result, run_name, model_name, params):
    X, model = result["input_example"], result["model"]
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics({k: float(v) for k, v in result["metrics"].items()})
        signature = infer_signature(X, model.predict_proba(X)[:, 1])
        mlflow.sklearn.log_model(model, artifact_path="model", signature=signature,
                                 input_example=X.head(5), registered_model_name=model_name)
    print(f"Registered {model_name} ({result['name']})")


register(pregame, "pregame-winner", "nba-pregame-winner",
         {"algorithm": pregame["name"], "fit_on": "2007-08 to 2025-26", "features": ", ".join(FEATURES)})
register(live, "live-win-probability", "nba-live-win-probability",
         {"algorithm": live["name"], "fit_on": "2010-11 to 2021-22", "snapshot_seconds": STEP})

## Save the report tables to NBA_Lakehouse (Delta) and keep a copy of the results in Files

In [ ]:
import shutil

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, FloatType


def save_table(path):
    pdf = pd.read_csv(path)
    for col in pdf.columns[pdf.dtypes == object]:
        pdf[col] = pdf[col].astype(object).where(pdf[col].notna(), None)
    sdf = spark.createDataFrame(pdf)
    for field in sdf.schema.fields:                         # NaN -> null, so Power BI sees blanks
        if isinstance(field.dataType, (DoubleType, FloatType)):
            sdf = sdf.withColumn(field.name, F.when(F.isnan(field.name), None).otherwise(F.col(field.name)))
    if "game_date" in pdf.columns:
        sdf = sdf.withColumn("game_date", F.to_date("game_date"))
    sdf.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(path.stem)
    print(f"{path.stem:24s} {len(pdf):>9,} rows")


for path in sorted((C.FABRIC / "tables").glob("*.csv")):
    save_table(path)

files = Path("/lakehouse/default/Files/nba")
for folder, name in ((C.TABLES, "results"), (C.FABRIC / "tables", "report_tables")):
    (files / name).mkdir(parents=True, exist_ok=True)
    for f in sorted(folder.glob("*.csv")):
        shutil.copyfile(f, files / name / f.name)       # file contents only: the OneLake mount keeps no Unix permissions
print("Copied result tables to Files/nba/")